# 03 — Particle–mesh gravity from scratch

We will build the PM loop in 2-D: deposit particles on a mesh, solve Poisson's equation, interpolate the force, and advance the particles with kick–drift–kick (KDK).

This is a dimensionless teaching model. It leaves out expansion, Hubble drag, cosmological time factors, and the physical normalization of gravity.

In [1]:
from pathlib import Path
import os, sys

ROOT = Path(os.path.abspath('.')).parent
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".mplconfig"))

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

SEED = 2602
EVIDENCE_DOMAIN = "analytic-fixture"
COLORS = {
    "blue": "#2D6A9F",
    "orange": "#E6862E",
    "green": "#3A8D72",
    "purple": "#7656A5",
    "red": "#C94C4C",
    "gray": "#626C78",
}
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

def savefig(fig, name):
    fig.text(0.99, 0.01, EVIDENCE_DOMAIN, ha="right", fontsize=7, color=COLORS["gray"])
    path = OUTPUT_DIR / name
    fig.savefig(path, bbox_inches="tight")
    return path

## 1. Density to force

We use cloud-in-cell (CIC) for both deposition and interpolation. In Fourier space,

$$
\nabla^2\phi=\delta,\qquad
\phi_{\mathbf k}=-\frac{\delta_{\mathbf k}}{k^2},\qquad
\mathbf a=-\nabla\phi.
$$

The $k=0$ density mode is the uniform background, so it produces no peculiar force.

In [2]:
def cic_stencil_2d(positions, nmesh, box_size):
    """Indices and weights of the four periodic CIC neighbours."""
    mesh_position = (positions % box_size) / (box_size / nmesh)
    lower = np.floor(mesh_position).astype(int)
    fraction = mesh_position - lower
    for offset in np.ndindex(2, 2):
        offset = np.asarray(offset)
        index = (lower + offset) % nmesh
        weight = np.prod(np.where(offset, fraction, 1 - fraction), axis=1)
        yield index[:, 0], index[:, 1], weight


def cic_deposit_2d(positions, nmesh, box_size, masses=None):
    masses = np.ones(len(positions)) if masses is None else np.asarray(masses)
    mesh = np.zeros((nmesh, nmesh))
    # TODO 1: scatter masses * weight onto the four neighbouring mesh sites.
    return mesh


def cic_interpolate_2d(vector_grid, positions, box_size):
    nmesh = vector_grid.shape[0]
    values = np.zeros((len(positions), vector_grid.shape[-1]))
    # TODO 2: use the same stencil to gather the grid values.
    raise NotImplementedError


def poisson_force_2d(delta, box_size):
    nmesh = delta.shape[0]
    k = 2 * np.pi * np.fft.fftfreq(nmesh, d=box_size / nmesh)
    kx, ky = np.meshgrid(k, k, indexing="ij")
    k2 = kx**2 + ky**2
    delta_k = np.fft.fftn(delta)
    # TODO 3: solve for phi_k at k>0 and return phi and -grad(phi).
    raise NotImplementedError

In [ ]:
# These three checks correspond directly to the three operators above.
box_size, nmesh = 1.0, 32
rng = np.random.default_rng(SEED)
test_positions = rng.uniform(0, box_size, size=(80, 2))
test_masses = rng.uniform(0.5, 1.5, size=len(test_positions))

deposited = cic_deposit_2d(test_positions, nmesh, box_size, test_masses)
mass_error = abs(deposited.sum() - test_masses.sum())

constant_grid = np.broadcast_to([1.2, -0.4], (nmesh, nmesh, 2)).copy()
gathered = cic_interpolate_2d(constant_grid, test_positions, box_size)
gather_error = np.max(np.abs(gathered - [1.2, -0.4]))

amplitude, mode = 0.4, 3
x = np.arange(nmesh) * box_size / nmesh
wave_number = 2 * np.pi * mode / box_size
delta_wave = amplitude * np.cos(wave_number * x[:, None]) * np.ones((1, nmesh))
_, force_wave = poisson_force_2d(delta_wave, box_size)
#expected_force = TODO: what's the expected force for a single-mode density wave? (calculate analytically given the equations and for cosine wave delta)
force_error = np.max(np.abs(force_wave[:, 0, 0] - expected_force))

assert mass_error < 1e-11
assert gather_error < 1e-12
assert force_error < 1e-12
print(f"mass conservation error: {mass_error:.2e}")
print(f"constant-field gather error: {gather_error:.2e}")
print(f"single-mode force error: {force_error:.2e}")

fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
ax.plot(x, expected_force, lw=2.5, color=COLORS["orange"], label="analytic")
ax.plot(x, force_wave[:, 0, 0], "o", ms=4, color=COLORS["blue"], label="PM")
ax.set(xlabel="x [toy units]", ylabel=r"$a_x$ [toy units]",
       title="Single-mode force")
ax.legend()
savefig(fig, "03_force_validation.png")
plt.show()

NotImplementedError: 

## 2. Add time evolution

KDK applies a half kick, a full drift, and a second half kick. The second kick must use the force at the new particle positions.

In [ ]:
def acceleration_at_particles(positions, nmesh, box_size):
    # compute mass overdensity, then force grid, then interpolate to particles to get acceleration
    # Return acceleration, delta
    raise NotImplementedError


def kdk_step(positions, velocities, dt, nmesh, box_size):
    acceleration, _ = acceleration_at_particles(positions, nmesh, box_size)
    # TODO 4: half-kick, drift, recompute the force, and half-kick again.
    # Return positions and velocities after the step.
    raise NotImplementedError

In [ ]:
# A smooth displaced lattice gives a readable initial condition.
nside = 32
axis = (np.arange(nside) + 0.5) / nside
qx, qy = np.meshgrid(axis, axis, indexing="ij")
displacement = np.stack([
    0.015 * np.sin(2 * np.pi * qx) + 0.006 * np.sin(4 * np.pi * (qx + qy)),
    0.015 * np.sin(2 * np.pi * qy) + 0.006 * np.sin(4 * np.pi * (qx - qy)),
], axis=-1)
positions_initial = (np.stack([qx, qy], axis=-1) + displacement).reshape(-1, 2)
velocities_initial = (0.3 * displacement).reshape(-1, 2)

positions = positions_initial.copy()
velocities = velocities_initial.copy()
trajectory = [positions.copy()]
nsteps, dt = 48, 0.05
for step in range(nsteps):
    positions, velocities = kdk_step(positions, velocities, dt, nmesh, box_size)
    if (step + 1) % 6 == 0:
        trajectory.append(positions.copy())
trajectory = np.asarray(trajectory)

_, delta_initial = acceleration_at_particles(positions_initial, nmesh, box_size)
_, delta_final = acceleration_at_particles(positions, nmesh, box_size)
limit = np.quantile(np.abs(np.r_[delta_initial.ravel(), delta_final.ravel()]), 0.99)
norm = mpl.colors.TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.7), constrained_layout=True)
image = axes[0].imshow(delta_initial.T, origin="lower", extent=[0, 1, 0, 1],
                       cmap="RdBu_r", norm=norm)
axes[0].set_title("initial density")
axes[1].imshow(delta_final.T, origin="lower", extent=[0, 1, 0, 1],
               cmap="RdBu_r", norm=norm)
axes[1].set_title(f"after {nsteps} KDK steps")

selected = np.where(np.all((positions_initial > 0.1) & (positions_initial < 0.9), axis=1))[0][::45]
for index in selected:
    axes[2].plot(trajectory[:, index, 0], trajectory[:, index, 1],
                 color=COLORS["green"], lw=1)
axes[2].scatter(positions_initial[selected, 0], positions_initial[selected, 1],
                s=12, color=COLORS["blue"], label="start")
axes[2].scatter(positions[selected, 0], positions[selected, 1],
                s=16, marker="x", color=COLORS["orange"], label="finish")
axes[2].set_title("selected trajectories")
axes[2].legend()

for ax in axes:
    ax.set(xlabel="x [toy units]", ylabel="y [toy units]",
           xlim=(0, 1), ylim=(0, 1), aspect="equal")
    ax.grid(False)
fig.colorbar(image, ax=axes[:2], label=r"overdensity $\delta$", shrink=0.8)
savefig(fig, "03_pm_evolution.png")
plt.show()

## Main points

- CIC smooths the density when depositing and smooths the force again when interpolating.
- The zero mode belongs to the background, not the peculiar force.
- Mesh resolution controls the force scale; the time step controls integration error.

The next notebook measures those last two errors separately.